In [1]:
from dotenv import load_dotenv
import os
from pathlib import Path
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

## Load test dataset

In [32]:
dotenv_path = "../.env"
dotenv_path = Path("../.env")
load_dotenv(dotenv_path=dotenv_path)

gemini_api_key = os.getenv("GEMINI_API_KEY2") # API KEY for baseline.

In [33]:
# Shuffle
test_df = pd.read_csv('../Splits/example_test/Mixed.csv')
test_df

,Unnamed: 0.1,writer,problem_id,submission_id,website,source,Unnamed: 0
0,0,AI,164A,NaN,https://codeforces.com,#include <bits/stdc++.h>\nusing namespace std;...,NaN
1,1,AI,NaN,NaN,https://codeforces.com,#include <bits/stdc++.h>\nusing namespace std;...,NaN
2,2,AI,o61_may08_estate,NaN,https://programming.in.th,#include <bits/stdc++.h>\nusing namespace std;...,1843.0
3,3,human,NaN,13646458.0,https://codeforces.com,//Template\n\n// By Anudeep :)\n//Includes\n#i...,NaN
4,4,AI,1032,NaN,https://programming.in.th,#include <bits/stdc++.h>\nusing namespace std;...,998.0
...,...,...,...,...,...,...,...
2995,2995,human,1000,306681.0,https://programming.in.th,#include <bits/stdc++.h>\nusing namespace std;...,NaN
2996,2996,human,1055,310229.0,https://programming.in.th,#include <bits/stdc++.h>\nusing namespace std;...,NaN
2997,2997,Human,NaN,NaN,https://firefly.gchan.moe,#include <bits/stdc++.h>\n\nusing namespace st...,NaN
2998,2998,AI,888F,NaN,https://codeforces.com,#include <bits/stdc++.h>\nusing namespace std;...,NaN


In [34]:
# Prompt URL
prompt_path = "./Prompts/PromptV1.txt"

with open(prompt_path,'r') as _prompt:
  prompt = _prompt.read()
prompt

'You are the best C++ code detector. Your job is to determine whether the competitive programming source code is AI-generated or written by a human.\n\n### Steps:\n1. **Read the source code carefully** to understand what does the code doing.\n2. **Identify the patterns of code** and analyze whter it\'s AI-generated or not, like checking unnesscary space, variables, new lines, or comments.\n3. **Determine if it\'s Human or AI** from the data you have.\n4. **From the reason** Answer if it AI or Human.\n\nYou must respond in JSON format, like this:\n```json\n{   \n    "reason": "Why this code is human-written or AI-generated"\n    "answer": "human" / "AI",  \n}\n```\n- You must not provide any other text, except JSON.\n- In the reason part, don\'t use symbol like " , ` , \\n, use only normal character.\nHere is C++ source code:\n'

In [35]:
test_df['prompt'] = prompt + "\n\n" + test_df['source']
test_df['prompt'][0]

'You are the best C++ code detector. Your job is to determine whether the competitive programming source code is AI-generated or written by a human.\n\n### Steps:\n1. **Read the source code carefully** to understand what does the code doing.\n2. **Identify the patterns of code** and analyze whter it\'s AI-generated or not, like checking unnesscary space, variables, new lines, or comments.\n3. **Determine if it\'s Human or AI** from the data you have.\n4. **From the reason** Answer if it AI or Human.\n\nYou must respond in JSON format, like this:\n```json\n{   \n    "reason": "Why this code is human-written or AI-generated"\n    "answer": "human" / "AI",  \n}\n```\n- You must not provide any other text, except JSON.\n- In the reason part, don\'t use symbol like " , ` , \\n, use only normal character.\nHere is C++ source code:\n\n\n#include <bits/stdc++.h>\nusing namespace std;\n\nint main() {\n    ios_base::sync_with_stdio(false);\n    cin.tie(nullptr);\n\n    int n, m;\n    cin >> n >>

In [36]:
BACKUP_PATH = "./Backups/Result-Gemini2.0.csv"

def save_responses(res):
    df = pd.DataFrame(res, columns=['back_up'])
    df.to_csv(BACKUP_PATH,index=False)

In [37]:
try:
    saved_df = pd.read_csv(BACKUP_PATH)
    answer = saved_df['back_up'].to_list()
except:
    answer = []

In [38]:
len(answer)

2948

In [39]:
from google import genai
from google.genai import types
import pathlib
import httpx
import time
from IPython.display import clear_output
client = genai.Client(api_key=gemini_api_key)
err_logs = []

err_cnt = 0

for index, row in test_df[len(answer):].iterrows():
  if index%50 == 0 and index != 0:
    save_responses(answer)
  while True:
    if err_cnt >= 10:
      break
    try:
      rprompt = row['prompt']
      response = client.models.generate_content(model="gemini-2.0-flash-lite",contents=[rprompt])
      s = response.text
      
      clear_output()
      print(f"ASKED : {index}")
      print(response.text)
      err_cnt = 0
      answer.append(s)
      break
    except Exception as e:
      err_cnt += 1
      err_logs.append((index,e))
      clear_output()
      print(f"ERROR at {index} : {e}")
      save_responses(answer)
      time.sleep(20)
save_responses(answer)

ASKED : 2999
```json
{
    "reason": "The code demonstrates a clear and logical approach to solving the problem. It uses nested loops and conditional statements in a way that is typical of human-written code. The structure and variable names are well-organized and suggest a human programmer's thought process. The code's comments, while minimal, are also consistent with human coding style.",
    "answer": "human"
}
```


In [40]:
save_responses(answer)

In [141]:
for idx, res in enumerate(answer):
   answer[idx] = res.text

In [41]:
answer

['```json\n{\n    "reason": "The code is well-structured with clear variable names and comments. It efficiently solves the problem by using adjacency lists and queues for graph traversal. The logic is sound and the code avoids unnecessary redundancies making it likely to be human-written.",\n    "answer": "human"\n}\n```',
 '```json\n{\n    "reason": "The code is well-structured uses meaningful variable names has comments and includes standard competitive programming practices like fast input output and clear logic, making it likely human-written.",\n    "answer": "human"\n}\n```',
 '```json\n{\n    "reason": "The code exhibits a well-structured approach to solving the problem involving graph traversal and group assignments. The use of a depth-first search (DFS) function with clear base cases and recursive steps, along with proper handling of group sizes indicates that it is human-written.",\n    "answer": "human"\n}\n```',
 '```json\n{\n    "reason": "The code contains a lot of commen

In [ ]:
answer[0]

'{\n    "answer": "human",\n    "reason": "The code demonstrates a clear understanding of graph traversal algorithms (breadth-first search) and their application to a specific problem. The variable naming is reasonably descriptive, and the code is well-structured with appropriate comments. The use of `ios_base::sync_with_stdio(false);` and `cin.tie(nullptr);` suggests a focus on performance, which is typical in competitive programming. The logic for determining reachability in both forward and backward directions, considering the constraints imposed by the values in the `f` vector (0, 1, and 2), is well-implemented and reflects a human\'s thought process in breaking down the problem."\n}'

In [42]:
import re
import json

result_n = len(answer)
answer_df = test_df.iloc[:result_n]
answer_df['answer_text'] = answer
pattern = re.compile(r'```json\s*(\{.*?\})\s*```', re.DOTALL)
answer_df['answer_dict'] = answer_df['answer_text'].apply(lambda x: x.lstrip('```json\n').rstrip('\n```'))


C:\Users\Pakin\AppData\Local\Temp\ipykernel_26412\446863633.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  answer_df['answer_text'] = answer
C:\Users\Pakin\AppData\Local\Temp\ipykernel_26412\446863633.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  answer_df['answer_dict'] = answer_df['answer_text'].apply(lambda x: x.lstrip('```json\n').rstrip('\n```'))


In [43]:
for idx,row in answer_df.iterrows():
    try:
        json.loads(row['answer_dict'])  
    except Exception as e:
        print(f"Error at {idx} :  {e}")
        print(f"Replace with {row['answer_dict'][:50]+row['answer_dict'][-5:]}")
        answer_df['answer_dict'][idx] = row['answer_dict'][:50]+row['answer_dict'][-5:]
        

In [44]:
answer_df['answer_dict'] = answer_df['answer_dict'].apply(lambda x: json.loads(x))
answer_df['model_answer'] = answer_df['answer_dict'].apply(lambda x: x['answer'])
answer_df['model_reason'] = answer_df['answer_dict'].apply(lambda x: x['reason'])
answer_df['verdict'] = answer_df['writer'] == answer_df['model_answer']
answer_df = answer_df[['writer','model_answer','verdict','model_reason','website','source']]

C:\Users\Pakin\AppData\Local\Temp\ipykernel_26412\3813125495.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  answer_df['answer_dict'] = answer_df['answer_dict'].apply(lambda x: json.loads(x))
C:\Users\Pakin\AppData\Local\Temp\ipykernel_26412\3813125495.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  answer_df['model_answer'] = answer_df['answer_dict'].apply(lambda x: x['answer'])
C:\Users\Pakin\AppData\Local\Temp\ipykernel_26412\3813125495.py:3: SettingWithCopyWarning: 
A value is trying to be set o

'. "\n}'

In [45]:
filtered_df = pd.DataFrame()
filtered_df[['Label','Model_Answer','Verdict','Reason','Website']] = answer_df[['writer','model_answer','verdict','model_reason','website']]

In [46]:
filtered_df

,Label,Model_Answer,Verdict,Reason,Website
0,AI,human,False,The code is well-structured with clear variabl...,https://codeforces.com
1,AI,human,False,The code is well-structured uses meaningful va...,https://codeforces.com
2,AI,human,False,The code exhibits a well-structured approach t...,https://programming.in.th
3,human,human,True,The code contains a lot of comments with speci...,https://codeforces.com
4,AI,human,False,The code solves a Sudoku validation problem. I...,https://programming.in.th
...,...,...,...,...,...
2995,human,human,True,The code has a clear structure with input proc...,https://programming.in.th
2996,human,human,True,The code is well-structured with clear variabl...,https://programming.in.th
2997,Human,human,False,The code is concise and well-structured. It de...,https://firefly.gchan.moe
2998,AI,human,False,The code demonstrates a dynamic programming ap...,https://codeforces.com


In [47]:
# I forgot about upper-lower case T-T
filtered_df['Label'] = filtered_df['Label'].replace('Human', 'human')
filtered_df['Verdict'] = filtered_df['Label']==filtered_df['Model_Answer']

In [48]:
filtered_df['Label'].value_counts(), filtered_df['Model_Answer'].value_counts()

(Label
 AI       1500
 human    1500
 Name: count, dtype: int64,
 Model_Answer
 human    2933
 AI         67
 Name: count, dtype: int64)

In [49]:
filtered_df

,Label,Model_Answer,Verdict,Reason,Website
0,AI,human,False,The code is well-structured with clear variabl...,https://codeforces.com
1,AI,human,False,The code is well-structured uses meaningful va...,https://codeforces.com
2,AI,human,False,The code exhibits a well-structured approach t...,https://programming.in.th
3,human,human,True,The code contains a lot of comments with speci...,https://codeforces.com
4,AI,human,False,The code solves a Sudoku validation problem. I...,https://programming.in.th
...,...,...,...,...,...
2995,human,human,True,The code has a clear structure with input proc...,https://programming.in.th
2996,human,human,True,The code is well-structured with clear variabl...,https://programming.in.th
2997,human,human,True,The code is concise and well-structured. It de...,https://firefly.gchan.moe
2998,AI,human,False,The code demonstrates a dynamic programming ap...,https://codeforces.com


In [50]:
filtered_df.to_csv("./Results/_2/Result-Gemini2.0.csv",index=False)

In [51]:
accuracy = filtered_df['Verdict'].value_counts(normalize=True).get(True, 0)
print(f"AUC : {accuracy}")

AUC : 0.491
